In [ ]:
# ====================================
# Import
# ====================================
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt

import joblib

from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier

# ====================================
# Definitions
# ====================================

def evaluate_pr(model, X_test, y_test):

    prob = model.predict_proba(X_test)[:, 1]

    precision, recall, thresholds = precision_recall_curve(
        y_test,
        prob
    )

    ap = average_precision_score(
        y_test,
        prob
    )

    print(ap)

    return precision, recall, ap

def plot_pr(baseline, precision, recall, ap):

    plt.plot(
        recall,
        precision,
        label=f"AP = {ap:.2f}"
    )

    plt.axhline(
        y = baseline,
        linestyle="--",
        label="Random baseline"
    )

    plt.xlabel(
        "Recall"
    )

    plt.ylabel(
        "Precision"
    )

    plt.title(
        "Precision-Recall Curve"
    )

    plt.legend()
    plt.show()


# ====================================
# Load
# ====================================

BASE_DIR = Path.cwd()

ini_file = BASE_DIR / "data" / "employees.csv"

df = pd.read_csv(ini_file)

# ====================================
# Data Cleaning
# ====================================

df["PerformanceScore"] = df["PerformanceScore"].fillna(
    df["PerformanceScore"].mode()[0]
)
df["Education"] = df["Education"].fillna(
    df["Education"].mode()[0]
)

# ====================================
# Creating unbalanced data
# ====================================

df["HighSalary"] = (
    df["Salary"] >= df["Salary"].median()
)

df_false = df[
    df["HighSalary"] == False
]

df_true = df[
    df["HighSalary"] == True
]

df_true = df_true.sample(
    n=10,
    random_state=42
)

df = pd.concat(
    [
        df_false,
        df_true
    ]
)

# ====================================
# Feature Selection
# ====================================

X = df[[
    "PerformanceScore",
    "Education",
    "Department",
    "Experience"
]].copy()


# ====================================
# Feature Engineering
# ====================================

X["ExperienceScore"] = (
    X["Experience"] * X["PerformanceScore"]
)

# ====================================
# Target
# ====================================

y = df["HighSalary"]

# ====================================
# Train/Test Split
# ====================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ====================================
# Feature split
# ====================================

numeric_features = [
    "PerformanceScore",
    "Experience",
    "ExperienceScore"
]

categorical_features = [
    "Education",
    "Department"
]

# ====================================
# ColumnTransformer
# ====================================

preprocessor = ColumnTransformer([
    (
        "numeric",
        StandardScaler(),
        numeric_features
    ),
    (
        "categorical",
        OneHotEncoder(drop="first"), 
        categorical_features
    )
])

# ====================================
# Models
# ====================================

logistic_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        LogisticRegression(
            class_weight="balanced"
        )
    )
])

forest_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        RandomForestClassifier(
            max_depth=5,
            n_estimators=100,
            random_state=42
        )
    )
])

svm_model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        SVC(
            kernel="linear",
            probability=True
        )
    )
])



# ====================================
# Hard/Soft voting
# ====================================

voting_model = VotingClassifier(
    estimators=[
        ("logistic", logistic_model),
        ("forest", forest_model),
        ("svm", svm_model)
    ],
    voting="soft"
)

voting_model.fit(
    X_train,
    y_train
)

pred = voting_model.predict(
    X_test
)

# # ====================================
# # Print metrics
# # ====================================

# print(
#     accuracy_score(
#         y_test,
#         pred
#     )
# )

# print(
#     confusion_matrix(
#         y_test,
#         pred
#     )
# )

# print(
#     classification_report(
#         y_test,
#         pred
#     )
# )

# prob = voting_model.predict_proba(X_test)

# print(prob)


# ====================================
# Stacking
# ====================================

stacking_model = StackingClassifier(
    estimators=[
        ("logistic", logistic_model),
        ("forest", forest_model),
        ("svm", svm_model)
    ],
    final_estimator=LogisticRegression(
        class_weight="balanced"
    )
)

stacking_model.fit(
    X_train,
    y_train
)

pred = stacking_model.predict(
    X_test
)

# ====================================
# Print metrics
# ====================================

print(
    accuracy_score(
        y_test,
        pred
    )
)

print(
    confusion_matrix(
        y_test,
        pred
    )
)

print(
    classification_report(
        y_test,
        pred
    )
)

# ====================================
# Model saving
# ====================================

model_path = BASE_DIR / "models" / "voting_model.pkl"

joblib.dump(
    voting_model,
    model_path
)


0.9285714285714286
[[13  0]
 [ 1  0]]
              precision    recall  f1-score   support

       False       0.93      1.00      0.96        13
        True       0.00      0.00      0.00         1

    accuracy                           0.93        14
   macro avg       0.46      0.50      0.48        14
weighted avg       0.86      0.93      0.89        14

[[0.70987247 0.29012753]
 [0.60047252 0.39952748]
 [0.65675656 0.34324344]
 [0.82423781 0.17576219]
 [0.65613538 0.34386462]
 [0.82792602 0.17207398]
 [0.8259695  0.1740305 ]
 [0.78824728 0.21175272]
 [0.72713754 0.27286246]
 [0.72124409 0.27875591]
 [0.83604995 0.16395005]
 [0.80277333 0.19722667]
 [0.61236072 0.38763928]
 [0.73147573 0.26852427]]


c:\Users\Drtic123\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Drtic123\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Drtic123\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} 

['c:\\MyProject\\PythonApp\\23_ensemble_learning\\models\\voting_model.pkl']